In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import json
import glob
import pickle
import time
import csv
import pandas as pd
import networkx as nx
from pathlib import Path
from datetime import datetime
from openai import OpenAI
from google.colab import userdata

In [3]:
# annotator agreement with LLM
base = "/content/drive/MyDrive/06-Green Washing AI/formlar_gw_categories/"
scores_file ="/content/drive/MyDrive/06-Green Washing AI/formlar_gw_categories/all_scores_csv.csv"

In [4]:
df_raw = pd.read_csv(scores_file, sep=";")
df_raw.head()

,Item_no,vagueness,misleading,concealment,overselling,irrelevance,vagueness.1,misleading.1,concealment.1,overselling.1,...,vagueness.5,misleading.5,concealment.5,overselling.5,irrelevance.5,vagueness.6,misleading.6,concealment.6,overselling.6,irrelevance.6
0,1,2,0,2,1,0,1,1,2,1,...,2,0,1,1,0,2,2,2,1,0
1,2,2,0,2,2,0,1,1,2,1,...,1,0,1,0,0,2,1,2,2,0
2,3,3,0,2,2,1,2,1,2,1,...,2,0,1,1,0,1,0,1,1,0
3,4,2,0,0,3,0,1,2,2,1,...,1,0,1,1,0,2,3,2,3,0
4,5,3,0,3,1,3,1,0,1,1,...,1,0,0,0,3,3,2,1,2,3


In [5]:
!pip install krippendorff

In [6]:
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score
import krippendorff  #

# ── Parse ─────────────────────────────────────────────────────────────────────
from io import StringIO
df_raw = pd.read_csv(scores_file, sep=";")
df_raw.columns = ["item"] + [
    f"{rater}_{dim}"
    for rater in ["A", "B", "C", "D", "Fatih", "Merve", "LLM"]
    for dim in ["vagueness", "misleading", "concealment", "overselling", "irrelevance"]
]
df_raw = df_raw.set_index("item")

DIMS    = ["vagueness", "misleading", "concealment", "overselling", "irrelevance"]
HUMANS  = ["A", "B", "C", "D", "Fatih", "Merve"]
ALL     = HUMANS + ["LLM"]

# ── Helper: extract rater × item matrix for one dimension ─────────────────────
def dim_matrix(raters, dim):
    """Returns numpy array shape (n_raters, n_items) for krippendorff."""
    return np.array([df_raw[f"{r}_{dim}"].values for r in raters], dtype=float)

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. KRIPPENDORFF'S ALPHA — human raters only, per dimension
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("1. KRIPPENDORFF'S ALPHA — 6 Human Raters")
print("   (ordinal level — respects 0-3 scale order)")
print("=" * 60)
print(f"{'Dimension':<15} {'Alpha':>8}  {'Interpretation'}")
print("-" * 50)

alpha_results = {}
for dim in DIMS:
    matrix = dim_matrix(HUMANS, dim)
    alpha  = krippendorff.alpha(reliability_data=matrix, level_of_measurement="ordinal")
    alpha_results[dim] = alpha
    if alpha >= 0.80:
        interp = "Strong"
    elif alpha >= 0.67:
        interp = "Acceptable"
    else:
        interp = "Weak"
    print(f"{dim:<15} {alpha:>8.3f}  {interp}")

overall_alpha = np.mean(list(alpha_results.values()))
print(f"\n{'Overall (mean)':<15} {overall_alpha:>8.3f}")

# ══════════════════════════════════════════════════════════════════════════════
# 2. KRIPPENDORFF'S ALPHA — all 7 raters (humans + LLM), per dimension
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("2. KRIPPENDORFF'S ALPHA — 6 Humans + LLM")
print("=" * 60)
print(f"{'Dimension':<15} {'Alpha':>8}  {'Interpretation'}")
print("-" * 50)

for dim in DIMS:
    matrix = dim_matrix(ALL, dim)
    alpha  = krippendorff.alpha(reliability_data=matrix, level_of_measurement="ordinal")
    if alpha >= 0.80:
        interp = "Strong"
    elif alpha >= 0.67:
        interp = "Acceptable"
    else:
        interp = "Weak"
    print(f"{dim:<15} {alpha:>8.3f}  {interp}")

1. KRIPPENDORFF'S ALPHA — 6 Human Raters
   (ordinal level — respects 0-3 scale order)
Dimension          Alpha  Interpretation
--------------------------------------------------
vagueness          0.404  Weak
misleading         0.063  Weak
concealment       -0.042  Weak
overselling        0.246  Weak
irrelevance        0.608  Weak

Overall (mean)     0.256

2. KRIPPENDORFF'S ALPHA — 6 Humans + LLM
Dimension          Alpha  Interpretation
--------------------------------------------------
vagueness          0.418  Weak
misleading        -0.024  Weak
concealment       -0.010  Weak
overselling        0.196  Weak
irrelevance        0.601  Weak


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# 3. WEIGHTED COHEN'S KAPPA — each human vs LLM, per dimension
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("3. WEIGHTED COHEN'S KAPPA — Each Human vs LLM")
print("   (quadratic weights — penalises larger disagreements more)")
print("=" * 60)

kappa_records = []
for dim in DIMS:
    llm_scores = df_raw[f"LLM_{dim}"].values
    row = {"dimension": dim}
    for h in HUMANS:
        h_scores = df_raw[f"{h}_{dim}"].values
        try:
            k = cohen_kappa_score(h_scores, llm_scores, weights="quadratic",
                                  labels=[0, 1, 2, 3])
        except Exception:
            k = np.nan
        row[h] = round(k, 3)
    row["mean_kappa"] = round(np.nanmean([row[h] for h in HUMANS]), 3)
    kappa_records.append(row)

kappa_df = pd.DataFrame(kappa_records).set_index("dimension")
print(kappa_df.to_string())

print("\nInterpretation guide: <0.20 Slight | 0.21-0.40 Fair | 0.41-0.60 Moderate")
print("                       0.61-0.80 Substantial | >0.80 Almost perfect")



3. WEIGHTED COHEN'S KAPPA — Each Human vs LLM
   (quadratic weights — penalises larger disagreements more)
                 A      B      C      D  Fatih  Merve  mean_kappa
dimension                                                        
vagueness    0.491  0.377  0.315  0.505  0.621  0.476       0.464
misleading   0.000  0.278  0.023  0.011  0.000 -0.047       0.044
concealment -0.131  0.486 -0.325  0.171  0.126  0.171       0.083
overselling  0.113  0.000 -0.017  0.189  0.255 -0.228       0.052
irrelevance  0.653  0.759  0.782  0.921  0.327  0.767       0.702

Interpretation guide: <0.20 Slight | 0.21-0.40 Fair | 0.41-0.60 Moderate
                       0.61-0.80 Substantial | >0.80 Almost perfect


In [9]:


from sklearn.metrics import cohen_kappa_score

for dim in DIMS:
    llm = df_raw[f"LLM_{dim}"].values
    for h in HUMANS:
        h_scores = df_raw[f"{h}_{dim}"].values
        k_weighted   = cohen_kappa_score(h_scores, llm, weights="quadratic", labels=[0,1,2,3])
        k_unweighted = cohen_kappa_score(h_scores, llm, weights=None, labels=[0,1,2,3])
        print(f"{dim:<15} {h:<8} weighted={k_weighted:.3f}  unweighted={k_unweighted:.3f}")

vagueness       A        weighted=0.491  unweighted=0.477
vagueness       B        weighted=0.377  unweighted=0.114
vagueness       C        weighted=0.315  unweighted=0.357
vagueness       D        weighted=0.505  unweighted=0.283
vagueness       Fatih    weighted=0.621  unweighted=0.244
vagueness       Merve    weighted=0.476  unweighted=0.336
misleading      A        weighted=0.000  unweighted=0.000
misleading      B        weighted=0.278  unweighted=-0.141
misleading      C        weighted=0.023  unweighted=-0.226
misleading      D        weighted=0.011  unweighted=-0.063
misleading      Fatih    weighted=0.000  unweighted=0.000
misleading      Merve    weighted=-0.047  unweighted=-0.063
concealment     A        weighted=-0.131  unweighted=0.091
concealment     B        weighted=0.486  unweighted=0.116
concealment     C        weighted=-0.325  unweighted=-0.102
concealment     D        weighted=0.171  unweighted=0.216
concealment     Fatih    weighted=0.126  unweighted=-0.057
conce

In [24]:

# ══════════════════════════════════════════════════════════════════════════════
# 4. WEIGHTED COHEN'S KAPPA — each human vs Human, per dimension
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.metrics import cohen_kappa_score
from itertools import combinations

print("Mean quadratic weighted kappa — human pairs only")
for dim in DIMS:
    kappas = []
    for h1, h2 in combinations(HUMANS, 2):
        s1 = df_raw[f"{h1}_{dim}"].values
        s2 = df_raw[f"{h2}_{dim}"].values
        k = cohen_kappa_score(s1, s2, weights="quadratic", labels=[0,1,2,3])
        kappas.append(k)
    print(f"  {dim:<15} mean κ = {np.mean(kappas):.3f}")

Mean quadratic weighted kappa — human pairs only
  vagueness       mean κ = 0.490
  misleading      mean κ = nan
  concealment     mean κ = 0.010
  overselling     mean κ = 0.276
  irrelevance     mean κ = 0.668


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


In [11]:

# ══════════════════════════════════════════════════════════════════════════════
# 4. MEAN SCORE COMPARISON — human average vs LLM, per dimension
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("4. MEAN SCORE COMPARISON — Human Average vs LLM")
print("=" * 60)
print(f"{'Dimension':<15} {'Human Mean':>12} {'LLM Mean':>10} {'Difference':>12}")
print("-" * 52)

for dim in DIMS:
    human_scores = df_raw[[f"{h}_{dim}" for h in HUMANS]].values.flatten()
    llm_scores   = df_raw[f"LLM_{dim}"].values
    h_mean = np.mean(human_scores)
    l_mean = np.mean(llm_scores)
    print(f"{dim:<15} {h_mean:>12.3f} {l_mean:>10.3f} {l_mean - h_mean:>+12.3f}")

# ── Save results ──────────────────────────────────────────────────────────────
kappa_df.to_csv(base + "kappa_llm_vs_humans.csv")
pd.DataFrame(alpha_results, index=["alpha"]).to_csv("krippendorff_alpha.csv")
print("\n✓ Results saved to kappa_llm_vs_humans.csv and krippendorff_alpha.csv")


4. MEAN SCORE COMPARISON — Human Average vs LLM
Dimension         Human Mean   LLM Mean   Difference
----------------------------------------------------
vagueness              1.442      1.650       +0.208
misleading             0.250      1.450       +1.200
concealment            1.392      1.150       -0.242
overselling            1.083      1.600       +0.517
irrelevance            0.675      0.350       -0.325

✓ Results saved to kappa_llm_vs_humans.csv and krippendorff_alpha.csv


In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# 5. PERCENTAGE AGREEMENT — each human vs LLM, per dimension
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("3. PERCENTAGE AGREEMENT — Each Human vs LLM")
print("=" * 60)

N = len(df_raw)

for dim in DIMS:
    llm = df_raw[f"LLM_{dim}"].values
    print(f"\n  {dim.upper()}")
    print(f"  {'Rater':<8} {'Exact %':>9} {'Adjacent %':>12}")
    print(f"  {'-'*32}")
    exact_list = []
    adj_list   = []
    for h in HUMANS:
        h_scores = df_raw[f"{h}_{dim}"].values
        exact    = np.mean(h_scores == llm) * 100
        adjacent = np.mean(np.abs(h_scores - llm) <= 1) * 100
        exact_list.append(exact)
        adj_list.append(adjacent)
        print(f"  {h:<8} {exact:>8.1f}% {adjacent:>11.1f}%")
    print(f"  {'MEAN':<8} {np.mean(exact_list):>8.1f}% {np.mean(adj_list):>11.1f}%")


3. PERCENTAGE AGREEMENT — Each Human vs LLM

  VAGUENESS
  Rater      Exact %   Adjacent %
  --------------------------------
  A            60.0%        75.0%
  B            40.0%        90.0%
  C            50.0%        70.0%
  D            50.0%        90.0%
  Fatih        45.0%        95.0%
  Merve        55.0%        90.0%
  MEAN         50.0%        85.0%

  MISLEADING
  Rater      Exact %   Adjacent %
  --------------------------------
  A            25.0%        45.0%
  B            15.0%        75.0%
  C             5.0%        60.0%
  D            20.0%        55.0%
  Fatih        25.0%        45.0%
  Merve        20.0%        45.0%
  MEAN         18.3%        54.2%

  CONCEALMENT
  Rater      Exact %   Adjacent %
  --------------------------------
  A            25.0%        55.0%
  B            45.0%       100.0%
  C            30.0%        80.0%
  D            50.0%        95.0%
  Fatih        35.0%        95.0%
  Merve        40.0%        90.0%
  MEAN         37.5%      

In [14]:
#  ── Summary table across all dimensions ──────────────────────────────────────
print("\n" + "=" * 60)
print("3b. SUMMARY — Mean % Agreement (across all human-LLM pairs)")
print("=" * 60)
print(f"  {'Dimension':<15} {'Exact %':>9} {'Adjacent %':>12}")
print(f"  {'-'*38}")
for dim in DIMS:
    llm = df_raw[f"LLM_{dim}"].values
    exacts = [np.mean(df_raw[f"{h}_{dim}"].values == llm) * 100 for h in HUMANS]
    adjs   = [np.mean(np.abs(df_raw[f"{h}_{dim}"].values - llm) <= 1) * 100 for h in HUMANS]
    print(f"  {dim:<15} {np.mean(exacts):>8.1f}% {np.mean(adjs):>11.1f}%")


print("\n" + "=" * 110)
print("3d. EXACT AND ADJACENT AGREEMENT PER HUMAN-LLM PAIR — by dimension")
print("=" * 110)

# Header
print(f"\n  {'':8}", end="")
for dim in DIMS:
    print(f"  {dim.upper():^17}", end="")
print(f"  {'MEAN':^17}")

print(f"  {'Rater':<8}", end="")
for dim in DIMS:
    print(f"  {'Ex%':>7} {'Adj%':>8}", end="")
print(f"  {'Ex%':>7} {'Adj%':>8}")
print(f"  {'-'*108}")

pair_rows = []
for h in HUMANS:
    row = {"rater": h}
    ex_scores, adj_scores = [], []
    line = f"  {h:<8}"
    for dim in DIMS:
        h_s  = df_raw[f"{h}_{dim}"].values
        llm  = df_raw[f"LLM_{dim}"].values
        ex   = np.mean(h_s == llm) * 100
        adj  = np.mean(np.abs(h_s - llm) <= 1) * 100
        row[f"{dim}_ex"]  = ex
        row[f"{dim}_adj"] = adj
        ex_scores.append(ex)
        adj_scores.append(adj)
        line += f"  {ex:>6.1f}% {adj:>6.1f}%"
    row["mean_ex"]  = np.mean(ex_scores)
    row["mean_adj"] = np.mean(adj_scores)
    line += f"  {row['mean_ex']:>6.1f}% {row['mean_adj']:>6.1f}%"
    pair_rows.append(row)
    print(line)

# Column means
print(f"  {'-'*108}")
pr_df = pd.DataFrame(pair_rows)
line  = f"  {'MEAN':<8}"
for dim in DIMS:
    line += f"  {pr_df[f'{dim}_ex'].mean():>6.1f}% {pr_df[f'{dim}_adj'].mean():>6.1f}%"
line += f"  {pr_df['mean_ex'].mean():>6.1f}% {pr_df['mean_adj'].mean():>6.1f}%"
print(line)


3b. SUMMARY — Mean % Agreement (across all human-LLM pairs)
  Dimension         Exact %   Adjacent %
  --------------------------------------
  vagueness           50.0%        85.0%
  misleading          18.3%        54.2%
  concealment         37.5%        85.8%
  overselling         44.2%        78.3%
  irrelevance         68.3%        93.3%

3d. EXACT AND ADJACENT AGREEMENT PER HUMAN-LLM PAIR — by dimension

                VAGUENESS         MISLEADING         CONCEALMENT        OVERSELLING        IRRELEVANCE           MEAN       
  Rater         Ex%     Adj%      Ex%     Adj%      Ex%     Adj%      Ex%     Adj%      Ex%     Adj%      Ex%     Adj%
  ------------------------------------------------------------------------------------------------------------
  A           60.0%   75.0%    25.0%   45.0%    25.0%   55.0%    20.0%   75.0%    60.0%   95.0%    38.0%   69.0%
  B           40.0%   90.0%    15.0%   75.0%    45.0%  100.0%    50.0%   85.0%    65.0%   95.0%    43.0%   89.0%
  

In [15]:
#══════════════════════════════════════════════════════════════════════════════
# 5c. EXPECTED CHANCE AGREEMENT & BINOMIAL TEST — each dimension
# ══════════════════════════════════════════════════════════════════════════════
from scipy.stats import binomtest

print("\n" + "=" * 80)
print("3c. EXPECTED CHANCE AGREEMENT & BINOMIAL TEST")
print("    (one-sided: observed > expected by chance)")
print("=" * 80)
print(f"  {'Dimension':<15} {'Obs Ex%':>8} {'Exp Ex%':>8} {'p-exact':>10} "
      f"{'Obs Adj%':>9} {'Exp Adj%':>9} {'p-adj':>10}")
print(f"  {'-'*75}")

N = len(df_raw)
chance_results = {}

for dim in DIMS:
    llm = df_raw[f"LLM_{dim}"].values

    all_human = np.concatenate([df_raw[f"{h}_{dim}"].values for h in HUMANS])
    p_human   = np.array([(all_human == s).mean() for s in range(4)])
    p_llm     = np.array([(llm == s).mean() for s in range(4)])

    # ── Expected EXACT chance agreement ──────────────────────────────────────
    p_exp_exact = float(np.dot(p_human, p_llm))

    # ── Expected ADJACENT chance agreement ───────────────────────────────────
    # P(|human - llm| <= 1) under independence
    p_exp_adj = 0.0
    for s_h in range(4):
        for s_l in range(4):
            if abs(s_h - s_l) <= 1:
                p_exp_adj += p_human[s_h] * p_llm[s_l]

    # ── Observed values ───────────────────────────────────────────────────────
    n_total      = N * len(HUMANS)
    obs_exact    = np.mean([np.mean(df_raw[f"{h}_{dim}"].values == llm) for h in HUMANS])
    obs_adj      = np.mean([np.mean(np.abs(df_raw[f"{h}_{dim}"].values - llm) <= 1)
                            for h in HUMANS])
    n_agree_ex   = int(round(obs_exact * n_total))
    n_agree_adj  = int(round(obs_adj   * n_total))

    # ── Binomial tests ────────────────────────────────────────────────────────
    p_val_exact = binomtest(n_agree_ex,  n_total, p_exp_exact, alternative="greater").pvalue
    p_val_adj   = binomtest(n_agree_adj, n_total, p_exp_adj,   alternative="greater").pvalue

    def sig(p):
        return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."

    chance_results[dim] = dict(
        obs_exact=obs_exact, p_exp_exact=p_exp_exact, p_val_exact=p_val_exact,
        obs_adj=obs_adj,     p_exp_adj=p_exp_adj,     p_val_adj=p_val_adj,
        n_agree_ex=n_agree_ex, n_agree_adj=n_agree_adj, n_total=n_total
    )

    print(f"  {dim:<15} {obs_exact*100:>7.1f}% {p_exp_exact*100:>7.1f}% "
          f"{p_val_exact:>8.4f}{sig(p_val_exact):>4} "
          f"{obs_adj*100:>8.1f}% {p_exp_adj*100:>8.1f}% "
          f"{p_val_adj:>8.4f}{sig(p_val_adj):>4}")

print("\n  Significance: *** p<0.001  ** p<0.01  * p<0.05  n.s. = not significant")



3c. EXPECTED CHANCE AGREEMENT & BINOMIAL TEST
    (one-sided: observed > expected by chance)
  Dimension        Obs Ex%  Exp Ex%    p-exact  Obs Adj%  Exp Adj%      p-adj
  ---------------------------------------------------------------------------
  vagueness          50.0%    28.0%   0.0000 ***     85.0%     71.6%   0.0004 ***
  misleading         18.3%    24.6%   0.9587n.s.     54.2%     53.3%   0.4644n.s.
  concealment        37.5%    33.7%   0.2159n.s.     85.8%     82.1%   0.1739n.s.
  overselling        44.2%    32.9%   0.0066  **     78.3%     78.1%   0.5302n.s.
  irrelevance        68.3%    55.8%   0.0035  **     93.3%     73.2%   0.0000 ***

  Significance: *** p<0.001  ** p<0.01  * p<0.05  n.s. = not significant


In [16]:
# confidence interval

from statsmodels.stats.proportion import proportion_confint

for dim in DIMS:
    llm = df_raw[f"LLM_{dim}"].values
    all_agree = []
    for h in HUMANS:
        h_s = df_raw[f"{h}_{dim}"].values
        all_agree.extend((h_s == llm).tolist())

    n_agree = sum(all_agree)
    n_total = len(all_agree)
    obs = n_agree / n_total
    lo, hi = proportion_confint(n_agree, n_total, alpha=0.05, method="wilson")
    print(f"{dim:<15} {obs*100:.1f}% [{lo*100:.1f}%, {hi*100:.1f}%]")

vagueness       50.0% [41.2%, 58.8%]
misleading      18.3% [12.4%, 26.2%]
concealment     37.5% [29.4%, 46.4%]
overselling     44.2% [35.6%, 53.1%]
irrelevance     68.3% [59.6%, 76.0%]


In [17]:
from statsmodels.stats.proportion import proportion_confint

print("Adjacent agreement with 95% Wilson CI — human-LLM pairs")
print(f"{'Dimension':<15} {'Obs%':>7} {'95% CI':>20}")
print("-" * 45)

for dim in DIMS:
    llm = df_raw[f"LLM_{dim}"].values
    all_adj = []
    for h in HUMANS:
        h_s = df_raw[f"{h}_{dim}"].values
        all_adj.extend((np.abs(h_s - llm) <= 1).tolist())

    n_agree = sum(all_adj)
    n_total = len(all_adj)
    obs     = n_agree / n_total
    lo, hi  = proportion_confint(n_agree, n_total, alpha=0.05, method="wilson")
    print(f"{dim:<15} {obs*100:>6.1f}%  [{lo*100:.1f}%, {hi*100:.1f}%]")

Adjacent agreement with 95% Wilson CI — human-LLM pairs
Dimension          Obs%               95% CI
---------------------------------------------
vagueness         85.0%  [77.5%, 90.3%]
misleading        54.2%  [45.3%, 62.8%]
concealment       85.8%  [78.5%, 91.0%]
overselling       78.3%  [70.1%, 84.8%]
irrelevance       93.3%  [87.4%, 96.6%]


In [18]:
from sklearn.metrics import cohen_kappa_score
import numpy as np

def bootstrap_kappa_ci(s1, s2, n_boot=1000, alpha=0.05):
    n = len(s1)
    boot_kappas = []
    for _ in range(n_boot):
        idx = np.random.choice(n, n, replace=True)
        try:
            k = cohen_kappa_score(s1[idx], s2[idx],
                                  weights="quadratic", labels=[0,1,2,3])
            boot_kappas.append(k)
        except Exception:
            pass
    lo = np.percentile(boot_kappas, 100 * alpha / 2)
    hi = np.percentile(boot_kappas, 100 * (1 - alpha / 2))
    return lo, hi

In [19]:
np.random.seed(42)  # for reproducibility

print("Weighted kappa with 95% bootstrap CI — human-LLM pairs")
print(f"{'Dimension':<15} {'Rater':<8} {'Kappa':>7} {'95% CI':>20}")
print("-" * 55)

for dim in DIMS:
    llm = df_raw[f"LLM_{dim}"].values
    kappas = []
    for h in HUMANS:
        h_s = df_raw[f"{h}_{dim}"].values
        k = cohen_kappa_score(h_s, llm, weights="quadratic", labels=[0,1,2,3])
        lo, hi = bootstrap_kappa_ci(h_s, llm, n_boot=1000)
        kappas.append(k)
        print(f"{dim:<15} {h:<8} {k:>7.3f}  [{lo:.3f}, {hi:.3f}]")
    print(f"{dim:<15} {'MEAN':<8} {np.nanmean(kappas):>7.3f}")
    print()

Weighted kappa with 95% bootstrap CI — human-LLM pairs
Dimension       Rater      Kappa               95% CI
-------------------------------------------------------
vagueness       A          0.491  [0.064, 0.805]
vagueness       B          0.377  [-0.058, 0.671]
vagueness       C          0.315  [-0.048, 0.628]
vagueness       D          0.505  [0.111, 0.764]
vagueness       Fatih      0.621  [0.336, 0.800]
vagueness       Merve      0.476  [0.000, 0.803]
vagueness       MEAN       0.464

misleading      A          0.000  [0.000, 0.000]
misleading      B          0.278  [-0.094, 0.533]
misleading      C          0.023  [-0.240, 0.190]
misleading      D          0.011  [-0.181, 0.191]
misleading      Fatih      0.000  [0.000, 0.000]
misleading      Merve     -0.047  [-0.164, 0.000]
misleading      MEAN       0.044

concealment     A         -0.131  [-0.356, 0.110]
concealment     B          0.486  [0.235, 0.626]
concealment     C         -0.325  [-0.609, 0.000]
concealment     D       

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/usr/local

irrelevance     D          0.921  [nan, nan]
irrelevance     Fatih      0.327  [0.000, 0.684]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


irrelevance     Merve      0.767  [nan, nan]
irrelevance     MEAN       0.701



In [21]:
# HUMAN HUMAN SCORES

In [25]:
from itertools import combinations
from sklearn.metrics import cohen_kappa_score
import numpy as np

# ── Human-Human Exact Agreement ───────────────────────────────────────────────
print("=" * 60)
print("HUMAN-HUMAN EXACT AGREEMENT")
print("Mean across all C(6,2) = 15 rater pairs per dimension")
print("=" * 60)
print(f"{'Dimension':<15} {'Mean Exact%':>12} {'Min%':>8} {'Max%':>8}")
print("-" * 45)

hh_exact = {}
for dim in DIMS:
    pair_exacts = []
    for h1, h2 in combinations(HUMANS, 2):
        s1 = df_raw[f"{h1}_{dim}"].values
        s2 = df_raw[f"{h2}_{dim}"].values
        exact = np.mean(s1 == s2) * 100
        pair_exacts.append(exact)
    mean_ex = np.mean(pair_exacts)
    hh_exact[dim] = mean_ex
    print(f"{dim:<15} {mean_ex:>11.1f}% {min(pair_exacts):>7.1f}% "
          f"{max(pair_exacts):>7.1f}%")

print(f"\n{'Overall mean':<15} {np.mean(list(hh_exact.values())):>11.1f}%")

HUMAN-HUMAN EXACT AGREEMENT
Mean across all C(6,2) = 15 rater pairs per dimension
Dimension        Mean Exact%     Min%     Max%
---------------------------------------------
vagueness              43.3%    15.0%    85.0%
misleading             66.3%    45.0%   100.0%
concealment            34.0%     5.0%    75.0%
overselling            40.3%    15.0%    75.0%
irrelevance            66.0%    35.0%    90.0%

Overall mean           50.0%


In [28]:
# ── Human-Human Adjacent Agreement ───────────────────────────────────────────
print("=" * 60)
print("HUMAN-HUMAN ADJACENT AGREEMENT")
print("Mean across all C(6,2) = 15 rater pairs per dimension")
print("=" * 60)
print(f"{'Dimension':<15} {'Mean Adj%':>12} {'Min%':>8} {'Max%':>8}")
print("-" * 45)

for dim in DIMS:
    pair_adjs = []
    for h1, h2 in combinations(HUMANS, 2):
        s1 = df_raw[f"{h1}_{dim}"].values
        s2 = df_raw[f"{h2}_{dim}"].values
        adj = np.mean(np.abs(s1 - s2) <= 1) * 100
        pair_adjs.append(adj)
    mean_adj = np.mean(pair_adjs)
    print(f"{dim:<15} {mean_adj:>11.1f}% {min(pair_adjs):>7.1f}% "
          f"{max(pair_adjs):>7.1f}%")

print(f"\n{'Overall mean':<15} {np.mean([np.mean([np.mean(np.abs(df_raw[f'{h1}_{dim}'].values - df_raw[f'{h2}_{dim}'].values) <= 1) * 100 for h1, h2 in combinations(HUMANS, 2)]) for dim in DIMS]):>11.1f}%")

HUMAN-HUMAN ADJACENT AGREEMENT
Mean across all C(6,2) = 15 rater pairs per dimension
Dimension          Mean Adj%     Min%     Max%
---------------------------------------------
vagueness              83.3%    35.0%   100.0%
misleading             95.0%    85.0%   100.0%
concealment            78.0%    30.0%   100.0%
overselling            91.7%    70.0%   100.0%
irrelevance            89.0%    70.0%   100.0%

Overall mean           87.4%


In [26]:
# ── Human-Human Weighted Kappa ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("HUMAN-HUMAN WEIGHTED KAPPA (quadratic, chance def. 2)")
print("Mean across all C(6,2) = 15 rater pairs per dimension")
print("=" * 60)
print(f"{'Dimension':<15} {'Mean κ':>8} {'Min κ':>8} {'Max κ':>8}")
print("-" * 42)

hh_kappa = {}
for dim in DIMS:
    pair_kappas = []
    for h1, h2 in combinations(HUMANS, 2):
        s1 = df_raw[f"{h1}_{dim}"].values
        s2 = df_raw[f"{h2}_{dim}"].values
        try:
            k = cohen_kappa_score(s1, s2, weights="quadratic",
                                  labels=[0, 1, 2, 3])
            pair_kappas.append(k)
        except Exception:
            pass
    mean_k = np.nanmean(pair_kappas)
    hh_kappa[dim] = mean_k
    print(f"{dim:<15} {mean_k:>8.3f} {min(pair_kappas):>8.3f} "
          f"{max(pair_kappas):>8.3f}")


HUMAN-HUMAN WEIGHTED KAPPA (quadratic, chance def. 2)
Mean across all C(6,2) = 15 rater pairs per dimension
Dimension         Mean κ    Min κ    Max κ
------------------------------------------
vagueness          0.490    0.211    0.884
misleading         0.082   -0.070    0.500
concealment        0.010   -0.310    0.543
overselling        0.276   -0.152    0.613
irrelevance        0.668    0.219    0.938


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


In [27]:
# ── Pairwise detail ───────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PAIRWISE DETAIL — all 15 pairs per dimension")
print("=" * 60)

for dim in DIMS:
    print(f"\n  {dim.upper()}")
    print(f"  {'Pair':<15} {'Exact%':>8} {'κ':>8}")
    print(f"  {'-'*33}")
    for h1, h2 in combinations(HUMANS, 2):
        s1 = df_raw[f"{h1}_{dim}"].values
        s2 = df_raw[f"{h2}_{dim}"].values
        exact = np.mean(s1 == s2) * 100
        try:
            k = cohen_kappa_score(s1, s2, weights="quadratic",
                                  labels=[0, 1, 2, 3])
        except Exception:
            k = float("nan")
        print(f"  {h1+' vs '+h2:<15} {exact:>7.1f}% {k:>8.3f}")


PAIRWISE DETAIL — all 15 pairs per dimension

  VAGUENESS
  Pair              Exact%        κ
  ---------------------------------
  A vs B             15.0%    0.283
  A vs C             25.0%    0.223
  A vs D             40.0%    0.294
  A vs Fatih         20.0%    0.313
  A vs Merve         30.0%    0.322
  B vs C             50.0%    0.615
  B vs D             70.0%    0.767
  B vs Fatih         35.0%    0.450
  B vs Merve         85.0%    0.860
  C vs D             45.0%    0.433
  C vs Fatih         35.0%    0.211
  C vs Merve         50.0%    0.545
  D vs Fatih         30.0%    0.585
  D vs Merve         85.0%    0.884
  Fatih vs Merve     35.0%    0.558

  MISLEADING
  Pair              Exact%        κ
  ---------------------------------
  A vs B             50.0%    0.000
  A vs C             50.0%    0.000
  A vs D             75.0%    0.000
  A vs Fatih        100.0%      nan
  A vs Merve         95.0%    0.000
  B vs C             55.0%    0.438
  B vs D             50.0% 

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


In [29]:
print("Score distribution (%) per dimension per human rater")
print("=" * 65)

for dim in DIMS:
    print(f"\n  {dim.upper()}")
    print(f"  {'Rater':<8} {'0':>8} {'1':>8} {'2':>8} {'3':>8}")
    print(f"  {'-'*40}")
    for h in HUMANS:
        scores = df_raw[f"{h}_{dim}"].values
        counts = [(scores == s).mean() * 100 for s in range(4)]
        print(f"  {h:<8} {counts[0]:>7.0f}% {counts[1]:>7.0f}% "
              f"{counts[2]:>7.0f}% {counts[3]:>7.0f}%")
    # Overall mean distribution
    all_h = np.concatenate([df_raw[f"{h}_{dim}"].values for h in HUMANS])
    counts = [(all_h == s).mean() * 100 for s in range(4)]
    print(f"  {'MEAN':<8} {counts[0]:>7.0f}% {counts[1]:>7.0f}% "
          f"{counts[2]:>7.0f}% {counts[3]:>7.0f}%")

Score distribution (%) per dimension per human rater

  VAGUENESS
  Rater           0        1        2        3
  ----------------------------------------
  A             10%      10%      20%      60%
  B              5%      65%      20%      10%
  C             45%      40%       5%      10%
  D             10%      50%      25%      15%
  Fatih         25%      25%      40%      10%
  Merve          5%      60%      25%      10%
  MEAN          17%      42%      22%      19%

  MISLEADING
  Rater           0        1        2        3
  ----------------------------------------
  A            100%       0%       0%       0%
  B             50%      35%      15%       0%
  C             50%      50%       0%       0%
  D             75%      20%       5%       0%
  Fatih        100%       0%       0%       0%
  Merve         95%       5%       0%       0%
  MEAN          78%      18%       3%       0%

  CONCEALMENT
  Rater           0        1        2        3
  ------------------